In [1]:
from nemo.collections.tts.models import T5TTS_Model
from nemo.collections.tts.data.text_to_speech_dataset import T5TTSDataset, DatasetSample
from omegaconf.omegaconf import OmegaConf, open_dict
import torch
import os
import soundfile as sf
from IPython.display import display, Audio
import os
import numpy as np
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="1"

[NeMo W 2025-01-24 15:55:08 nemo_logging:361] /usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
      from .autonotebook import tqdm as notebook_tqdm
    


## Set checkpoint and other file paths on your machine

In [2]:
hparams_file = "/datap/misc/duplexcheckpoints/blackwell/duplex_blackwell_medium_decoder_noexpresso_onlyphonemeFT_hparams.yaml"
checkpoint_file = "/datap/misc/duplexcheckpoints/blackwell/duplex_blackwell_medium_decoder_withTC_fromroycheckpoint_lowsestvalloss.ckpt"
codecmodel_path = "/datap/misc/checkpoints/AudioCodec_21Hz_no_eliz.nemo"
out_dir = "/datap/misc/t5tts_inference_notebook_samples"
if not os.path.exists(out_dir):
    os.makedirs(out_dir)

dummy_audio_filepath = os.path.join(out_dir, "dummy_audio.wav")
sf.write(dummy_audio_filepath, np.zeros(22050 * 3), 22050)

## Load Model

In [3]:
model_cfg = OmegaConf.load(hparams_file).cfg

with open_dict(model_cfg):
    model_cfg.codecmodel_path = codecmodel_path
    if hasattr(model_cfg, 'text_tokenizer'):
        # Backward compatibility for models trained with absolute paths in text_tokenizer
        model_cfg.text_tokenizer.g2p.phoneme_dict = "scripts/tts_dataset_files/ipa_cmudict-0.7b_nv23.01.txt"
        model_cfg.text_tokenizer.g2p.heteronyms = "scripts/tts_dataset_files/heteronyms-052722"
        model_cfg.text_tokenizer.g2p.phoneme_probability = 1.0
    model_cfg.train_ds = None
    model_cfg.validation_ds = None


model = T5TTS_Model(cfg=model_cfg)
# Load weights from checkpoint file
print("Loading weights from checkpoint")
ckpt = torch.load(checkpoint_file)
model.load_state_dict(ckpt['state_dict'])
print("Loaded weights.")

if model_cfg.t5_decoder.pos_emb == "learnable":
    if (model_cfg.t5_decoder.use_flash_self_attention) is False and (model_cfg.t5_decoder.use_flash_self_attention is False):
        print("Using kv cache for inference.")
        model.use_kv_cache_for_inference = True

model.cuda()
model.eval()

[NeMo W 2025-01-24 15:55:13 experimental:26] `<class 'nemo.collections.tts.g2p.models.i18n_ipa.IpaG2p'>` is experimental and not ready for production yet. Use at your own risk.
[NeMo W 2025-01-24 15:55:14 i18n_ipa:124] apply_to_oov_word=None, This means that some of words will remain unchanged if they are not handled by any of the rules in self.parse_one_word(). This may be intended if phonemes and chars are both valid inputs, otherwise, you may see unexpected deletions in your input.
[NeMo W 2025-01-24 15:55:14 experimental:26] `<class 'nemo.collections.common.tokenizers.text_to_speech.tts_tokenizers.IPATokenizer'>` is experimental and not ready for production yet. Use at your own risk.
[NeMo W 2025-01-24 15:55:14 experimental:26] `<class 'nemo.collections.tts.g2p.models.i18n_ipa.IpaG2p'>` is experimental and not ready for production yet. Use at your own risk.
[NeMo W 2025-01-24 15:55:15 i18n_ipa:124] apply_to_oov_word=None, This means that some of words will remain unchanged if they 

[NeMo I 2025-01-24 15:55:32 audio_codec:94] Vector quantizer does not support commit loss.
[NeMo I 2025-01-24 15:55:38 features:305] PADDING: 1
[NeMo I 2025-01-24 15:55:44 features:305] PADDING: 1
[NeMo I 2025-01-24 15:55:44 features:305] PADDING: 1
[NeMo I 2025-01-24 15:55:44 features:305] PADDING: 1
[NeMo I 2025-01-24 15:55:44 features:305] PADDING: 1
[NeMo I 2025-01-24 15:55:44 features:305] PADDING: 1
[NeMo I 2025-01-24 15:55:44 features:305] PADDING: 1
[NeMo I 2025-01-24 15:55:45 save_restore_connector:275] Model AudioCodecModel was successfully restored from /datap/misc/checkpoints/AudioCodec_21Hz_no_eliz.nemo.
Loading weights from checkpoint
Loaded weights.


T5TTS_Model(
  (audio_embeddings): ModuleList(
    (0-7): 8 x Embedding(2048, 1536)
  )
  (text_embedding): Embedding(106339, 1536)
  (t5_encoder): TransformerStack(
    (dropout): Dropout(p=0.1, inplace=False)
    (norm_out): LayerNorm()
    (layers): ModuleList(
      (0-5): 6 x TransformerBlock(
        (norm_self): LayerNorm()
        (self_attention): Attention(
          (qkv_net): Linear(in_features=1536, out_features=4608, bias=False)
          (o_net): Linear(in_features=1536, out_features=1536, bias=False)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (layer_scale_self_attn): Identity()
        (norm_pos_ff): LayerNorm()
        (pos_ff): PositionwiseConvFF(
          (non_linearity): GELU(approximate='tanh')
          (proj): ConvNorm(
            (conv): Conv1d(1536, 6144, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
          )
          (o_net): ConvNorm(
            (conv): Conv1d(6144, 1536, kernel_size=(3,), stride=(1,), padding=(1,),

In [4]:
test_dataset = T5TTSDataset(
    dataset_meta={},
    sample_rate=model_cfg.sample_rate,
    min_duration=0.5,
    max_duration=20,
    codec_model_downsample_factor=model_cfg.codec_model_downsample_factor,
    bos_id=model.bos_id,
    eos_id=model.eos_id,
    context_audio_bos_id=model.context_audio_bos_id,
    context_audio_eos_id=model.context_audio_eos_id,
    audio_bos_id=model.audio_bos_id,
    audio_eos_id=model.audio_eos_id,
    num_audio_codebooks=model_cfg.num_audio_codebooks,
    prior_scaling_factor=None,
    load_cached_codes_if_available=True,
    dataset_type='test',
    tokenizer_config=None,
    load_16khz_audio=model.model_type == 'single_encoder_sv_tts',
    use_text_conditioning_tokenizer=model.use_text_conditioning_encoder,
    pad_context_text_to_max_duration=model.pad_context_text_to_max_duration,
    context_duration_min=model.cfg.get('context_duration_min', 5.0),
    context_duration_max=model.cfg.get('context_duration_max', 5.0),
)
test_dataset.text_tokenizer, test_dataset.text_conditioning_tokenizer = model._setup_tokenizers(model.cfg, mode='test')

[NeMo W 2025-01-24 15:56:08 experimental:26] `<class 'nemo.collections.tts.g2p.models.i18n_ipa.IpaG2p'>` is experimental and not ready for production yet. Use at your own risk.
[NeMo W 2025-01-24 15:56:09 i18n_ipa:124] apply_to_oov_word=None, This means that some of words will remain unchanged if they are not handled by any of the rules in self.parse_one_word(). This may be intended if phonemes and chars are both valid inputs, otherwise, you may see unexpected deletions in your input.
[NeMo W 2025-01-24 15:56:09 experimental:26] `<class 'nemo.collections.common.tokenizers.text_to_speech.tts_tokenizers.IPATokenizer'>` is experimental and not ready for production yet. Use at your own risk.
[NeMo W 2025-01-24 15:56:09 experimental:26] `<class 'nemo.collections.tts.g2p.models.i18n_ipa.IpaG2p'>` is experimental and not ready for production yet. Use at your own risk.
[NeMo W 2025-01-24 15:56:10 i18n_ipa:124] apply_to_oov_word=None, This means that some of words will remain unchanged if they 

### Set dialogues to generate

Each item in the list can be a single-turn or a multi-turn dialogue.

[SPK-BWL-B-F] is the speaker tag for female speaker and [SPK-BWL-B-M] is the speaker tag for male speaker.

ChatGPT prompt that can generate something like this:

```
Generate dialogues for a 30 second podcast about <TOPIC>.
The conversation should be between a male and female speaker formatted as follows:
[SPK-BWL-B-F] Sentence by a female speaker
[SPK-BWL-B-M] Sentence by a male speaker
where [SPK-BWL-B-F] and [SPK-BWL-B-M] indicate speaker tags. Dont have any quotation marks in the text, and if there are any numbers spell them out. Basically, keep the text normalized suitable for a TTS model. Keep the conversation fun and engaging with the speakers talking and responding to each other. 
```


In [11]:
dialogues = ['[SPK-BWL-B-M] is brandy melville a person? ',
 '[SPK-BWL-B-F] No. Brandy Melville is a retail department store . ',
 '[SPK-BWL-B-M] what is it selling? ',
 '[SPK-BWL-B-F] It is an Italian brand which sells clothing and fashion accessories. Its products are targeting teenage girls and young women.']
dialogues = [d for d in dialogues if len(d) > 0]

### Generation

Below code generates 4 samples for each item in the dialogues list. Then asks, which one you like the best (index 0,1,2 or 3) and adds that to the already generated dialogue. You may modify the below code to automate and just select the first generation if you dont want to do this manually. After every dialogue it also plays the combined dialogues until that item.

In [12]:
from pydub import AudioSegment

def play_combined_audio(audio_files):
    combined_audio = AudioSegment.empty()
    for file in audio_files:
        audio = AudioSegment.from_file(file)
        combined_audio += audio
    output_path = os.path.join(out_dir, "combined_audio.wav")
    combined_audio.export(output_path, format="wav")
    display(Audio(output_path))
    return output_path
    


context_path = None
prev_context_duration = 5.0
generated_audios = []
for didx, dialogue in enumerate(dialogues):
    audio_dir = "/"
    entry = {
        "audio_filepath": dummy_audio_filepath,
        "duration": 3.0,
        "text": dialogue,
        "speaker": "dummy",
    }
    if didx == 0:
        entry["context_text"] = "MIXED SPEECH TTS"
    else:
        entry["context_text"] = "MIXED SPEECH TTS"
#         entry['context_audio_filepath'] = context_path
#         entry['context_audio_duration'] = prev_context_duration
        
        
        
    data_sample = DatasetSample(
        dataset_name="sample",
        manifest_entry=entry,
        audio_dir=audio_dir,
        feature_dir=audio_dir,
        text=entry['text'],
        speaker=None,
        speaker_index=0,
        tokenizer_names=["english_phoneme"]
    )
    test_dataset.data_samples = [data_sample]

    test_data_loader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=1,
        collate_fn=test_dataset.collate_fn,
        num_workers=0,
        shuffle=False
    )
    
    
    item_idx = 0
    for bidx, batch in enumerate(test_data_loader):
        print("Processing batch {} out of {}".format(bidx, len(test_data_loader)))
        model.t5_decoder.reset_cache(use_cache=True)
        batch_cuda ={}
        for key in batch:
            if isinstance(batch[key], torch.Tensor):
                batch_cuda[key] = batch[key].cuda()
            else:
                batch_cuda[key] = batch[key]
        import time
        
        candidates = []
        for try_idx in range(4):
            st = time.time()
            predicted_audio, predicted_audio_lens, _, _ = model.infer_batch(
                batch_cuda, 
                max_decoder_steps=500, 
                temperature=0.6, 
                topk=80, 
                use_cfg=True, 
                cfg_scale=1.6
            )
            print("generation time", time.time() - st)
            for idx in range(predicted_audio.size(0)):
                predicted_audio_np = predicted_audio[idx].float().detach().cpu().numpy()
                predicted_audio_np = predicted_audio_np[:predicted_audio_lens[idx]]
                audio_path = os.path.join(out_dir, f"predicted_audio_{try_idx}_{didx}_{item_idx}.wav")
                sf.write(audio_path, predicted_audio_np, model.cfg.sample_rate)
                print("Dialogue:", item_idx, "Candidate: ", try_idx)
                display(Audio(audio_path))
                candidates.append(audio_path)
        
        user_input = input("Enter Candidate number that sounds the best:").strip().lower()
        selected_audio_idx = int(user_input)
        audio_path = candidates[selected_audio_idx]
        item_idx += 1
        generated_audios.append(audio_path)
        print("Podcast generated until now:")
        combined_audio_path = play_combined_audio(generated_audios)
        last_gen_audio = AudioSegment.from_file(combined_audio_path)
        if len(last_gen_audio) > 5000:
            last_5_seconds = last_gen_audio[-5000:]  # Duration is in milliseconds
            prev_context_duration = 5.0
        else:
            last_5_seconds = last_gen_audio
            prev_context_duration = len(last_gen_audio)/1000.0
        context_path = os.path.join(out_dir, "context.wav")
        last_5_seconds.export(context_path)

Processing batch 0 out of 1
Decoding timestep 0
Decoding timestep 20
End detected for item 0 at timestep 32
All ends reached
generation time 0.8141214847564697
Dialogue: 0 Candidate:  0


Decoding timestep 0
Decoding timestep 20
End detected for item 0 at timestep 30
All ends reached
generation time 0.6909811496734619
Dialogue: 0 Candidate:  1


Decoding timestep 0
Decoding timestep 20
Decoding timestep 40
End detected for item 0 at timestep 43
All ends reached
generation time 0.940403938293457
Dialogue: 0 Candidate:  2


Decoding timestep 0
Decoding timestep 20
End detected for item 0 at timestep 32
All ends reached
generation time 0.6708316802978516
Dialogue: 0 Candidate:  3


Enter Candidate number that sounds the best:0
Podcast generated until now:


...

Processing batch 0 out of 1
Decoding timestep 0
Decoding timestep 20
Decoding timestep 40
Decoding timestep 60
Decoding timestep 80
Decoding timestep 100
Decoding timestep 120
End detected for item 0 at timestep 134
All ends reached
generation time 2.7284045219421387
Dialogue: 0 Candidate:  0


Decoding timestep 0
Decoding timestep 20
Decoding timestep 40
Decoding timestep 60
Decoding timestep 80
Decoding timestep 100
Decoding timestep 120
End detected for item 0 at timestep 135
All ends reached
generation time 2.650308847427368
Dialogue: 0 Candidate:  1


Decoding timestep 0
Decoding timestep 20
Decoding timestep 40
Decoding timestep 60
Decoding timestep 80
Decoding timestep 100
Decoding timestep 120
End detected for item 0 at timestep 133
All ends reached
generation time 2.657282829284668
Dialogue: 0 Candidate:  2


Decoding timestep 0
Decoding timestep 20
Decoding timestep 40
Decoding timestep 60
Decoding timestep 80
Decoding timestep 100
Decoding timestep 120
Decoding timestep 140
End detected for item 0 at timestep 140
All ends reached
generation time 2.794222354888916
Dialogue: 0 Candidate:  3


Enter Candidate number that sounds the best:0
Podcast generated until now:


In [ ]:
len(last_gen_audio)